# Official vCache on the SemanticCacheController workload

In [17]:
!git clone https://github.com/jg-eno/SemanticCacheController
%cd SemanticCacheController

Cloning into 'SemanticCacheController'...
remote: Enumerating objects: 238, done.
remote: Counting objects: 100% (238/238), done.
remote: Compressing objects: 100% (162/162), done.
remote: Total 238 (delta 110), reused 194 (delta 68), pack-reused 0 (from 0)
Receiving objects: 100% (238/238), 1.80 MiB | 23.90 MiB/s, done.
Resolving deltas: 100% (110/110), done.
/content/SemanticCacheController/SemanticCacheController/SemanticCacheController


In [3]:
#1)path
import os
REPO = "/content/SemanticCacheController"
assert os.path.exists(f"{REPO}/src/baseline_models/vcache_official.py"), "vcache_official.py not found - fix REPO or add the file"
os.environ["REPO"] = REPO
os.environ["VENV"] = "/content/venv311"
os.environ["VCACHE_DIR"] = "/content/vCache"
print("ok")

ok


In [4]:
%%bash
# 2) Python 3.11 venv (uv downloads 3.11 for us)
set -e
pip -q install uv
uv venv --python 3.11 "$VENV"
"$VENV/bin/python" --version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 89.1 MB/s eta 0:00:00
Python 3.11.17


Using CPython 3.11.17
Creating virtual environment at: /content/venv311
Activate with: source /content/venv311/bin/activate


In [5]:
%%bash
# 3) Official vCache at the pinned commit + this repo's dependencies (big install, several minutes)
set -e
[ -d "$VCACHE_DIR" ] || git clone https://github.com/vcache-project/vCache "$VCACHE_DIR"
git -C "$VCACHE_DIR" checkout 50694eebf0c870231030f73707cb6543d47f0411
uv pip install --python "$VENV/bin/python" -e "$VCACHE_DIR" scikit-learn gymnasium sentence-transformers
# vCache pins numpy 1.26, so use pyarrow/datasets versions that still support numpy 1.x
uv pip install --python "$VENV/bin/python" "pyarrow<19" "datasets<4"
"$VENV/bin/python" -c "import numpy, pyarrow, datasets, sentence_transformers; print('numpy', numpy.__version__, '| pyarrow', pyarrow.__version__, '| sentence_transformers OK')"
"$VENV/bin/python" -c "import vcache; print('vcache imports OK')"

numpy 1.26.4 | pyarrow 18.1.0 | sentence_transformers OK
INFO 10-19 20:27:05 [__init__.py:216] Automatically detected platform cuda.
vcache imports OK


Cloning into '/content/vCache'...
Note: switching to '50694eebf0c870231030f73707cb6543d47f0411'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 50694ee Add resource/throughput benchmark metrics and offline CI smoke coverage (#99)
Using Python 3.11.17 environment at: /content/venv311
Resolved 230 packages in 6.74s
Prepared 229 packages in 1m 54s
Installed 230 packages in 1.25s
 + accelerate==1.15.0
 + agent-detector==2.0.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiohttp-retry==2.9.1
 + aiosignal==1.4.0

In [14]:
%%bash
# 4) Adapter unit tests (stand-ins, no official policy) - should all pass
cd "$REPO"
"$VENV/bin/python" -m unittest src.tests.test_vcache_official_adapter src.tests.test_vcache_hybrid -v 2>&1 | tail -25

/content/venv311/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/content/venv311/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/content/venv311/lib/python3.11/site-packages/sklearn/

In [7]:
%%bash
# 5) Check every import the adapter needs before the real run
cd "$REPO"
"$VENV/bin/python" - <<'PY'
from vcache.config import VCacheConfig
from vcache.inference_engine.inference_engine import InferenceEngine
from vcache.main import VCache
from vcache.vcache_core.cache.embedding_engine.embedding_engine import EmbeddingEngine
from vcache.vcache_core.cache.embedding_store.embedding_metadata_storage.strategies.in_memory import InMemoryEmbeddingMetadataStorage
from vcache.vcache_core.cache.embedding_store.vector_db.strategies.hnsw_lib import HNSWLibVectorDB
from vcache.vcache_core.cache.eviction_policy.strategies.no_eviction import NoEvictionPolicy
from vcache.vcache_core.similarity_evaluator.similarity_evaluator import SimilarityEvaluator
from vcache.vcache_policy.strategies.verified import VerifiedDecisionPolicy
print("all official imports OK")
PY

INFO 10-19 20:27:20 [__init__.py:216] Automatically detected platform cuda.
all official imports OK


In [8]:
%%bash
# 6) Run the official policy on the example workload (it mostly explores on 96 requests - expected)
cd "$REPO"
"$VENV/bin/python" -m src.baseline_models.vcache_official \
  --workload src/datasets/example_workload.jsonl --deltas 0.01 0.02 0.05

INFO 10-19 20:27:34 [__init__.py:216] Automatically detected platform cuda.
{
  "seed": 0,
  "settings": {
    "workload": "src/datasets/example_workload.jsonl",
    "model": "sentence-transformers/all-MiniLM-L6-v2",
    "model_revision": null
  },
  "results": {
    "official_vCache_delta_0.01": {
      "requests": 96,
      "evaluable_labeled_requests": 96,
      "false_hit_rate": 0.0,
      "recomputation_rate": 0.4583333333333333,
      "mean_environment_reward": 0.4499999999999999,
      "explorations": 43
    },
    "official_vCache_delta_0.02": {
      "requests": 96,
      "evaluable_labeled_requests": 96,
      "false_hit_rate": 0.0,
      "recomputation_rate": 0.25,
      "mean_environment_reward": 0.6999999999999998,
      "explorations": 23
    },
    "official_vCache_delta_0.05": {
      "requests": 96,
      "evaluable_labeled_requests": 96,
      "false_hit_rate": 0.0,
      "recomputation_rate": 0.16666666666666666,
      "mean_environment_reward": 0.7999999999999998,
 

/content/venv311/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/content/venv311/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/content/venv311/lib/python3.11/site-packages/sklearn/

## Hybrid: official decision logic inside our environment
Same workload, our `SemanticCacheEnv` cache (capacity, eviction, threshold). Needs `benchmark/settings.json` and a gymnasium install (already in step 3).

In [9]:
%%bash
cd "$REPO"
"$VENV/bin/python" -W ignore -m unittest src.tests.test_vcache_hybrid -v 2>&1 | tail -20

test_entry_keeps_one_metadata_object (src.tests.test_vcache_hybrid.DecisionMappingTests.test_entry_keeps_one_metadata_object) ... ok
test_exploit_reuses_and_records_nothing (src.tests.test_vcache_hybrid.DecisionMappingTests.test_exploit_reuses_and_records_nothing) ... ok
test_explore_recomputes_and_records_a_correct_observation (src.tests.test_vcache_hybrid.DecisionMappingTests.test_explore_recomputes_and_records_a_correct_observation) ... ok
test_explore_records_an_incorrect_observation_for_a_different_intent (src.tests.test_vcache_hybrid.DecisionMappingTests.test_explore_records_an_incorrect_observation_for_a_different_intent) ... ok
test_recomputes_without_candidate_and_does_not_call_algorithm (src.tests.test_vcache_hybrid.DecisionMappingTests.test_recomputes_without_candidate_and_does_not_call_algorithm) ... ok
test_rejects_invalid_delta (src.tests.test_vcache_hybrid.DecisionMappingTests.test_rejects_invalid_delta) ... ok
test_unlabeled_request_records_nothing (src.tests.test_vcach

In [18]:
import json, os
p = os.path.join(os.environ["REPO"], "benchmark", "settings.json")
cfg = json.load(open(p))
print("before:", cfg)

target = cfg["settings"] if isinstance(cfg.get("settings"), dict) else cfg
target["capacity"] = 32
target["threshold"] = 0.85

json.dump(cfg, open(p, "w"), indent=2)
print("after: ", json.load(open(p)))

before: {'workload_path': 'src/datasets/example_workload.jsonl', 'capacity': 4, 'threshold': 0.99, 'window': 16}
after:  {'workload_path': 'src/datasets/example_workload.jsonl', 'capacity': 32, 'threshold': 0.85, 'window': 16}


In [19]:
%%bash
cd "$REPO"
"$VENV/bin/python" -W ignore -m src.benchmark --settings benchmark/settings.json \
  --methods benchmark/methods_vcache_hybrid.json --output benchmark/results_vcache_hybrid.json

INFO 10-19 20:39:18 [__init__.py:216] Automatically detected platform cuda.


In [20]:
import json, os
from google.colab import files
for name in ("results_vcache_official.json", "results_vcache_hybrid.json"):
    p = os.path.join(os.environ["REPO"], "benchmark", name)
    if os.path.exists(p):
        print(name); print(json.dumps(json.load(open(p)), indent=2)[:2000]); files.download(p)

results_vcache_official.json
{
  "seed": 0,
  "settings": {
    "workload": "src/datasets/example_workload.jsonl",
    "model": "sentence-transformers/all-MiniLM-L6-v2",
    "model_revision": null
  },
  "results": {
    "official_vCache_delta_0.01": {
      "requests": 96,
      "evaluable_labeled_requests": 96,
      "false_hit_rate": 0.0,
      "recomputation_rate": 0.4583333333333333,
      "mean_environment_reward": 0.4499999999999999,
      "explorations": 43
    },
    "official_vCache_delta_0.02": {
      "requests": 96,
      "evaluable_labeled_requests": 96,
      "false_hit_rate": 0.0,
      "recomputation_rate": 0.25,
      "mean_environment_reward": 0.6999999999999998,
      "explorations": 23
    },
    "official_vCache_delta_0.05": {
      "requests": 96,
      "evaluable_labeled_requests": 96,
      "false_hit_rate": 0.0,
      "recomputation_rate": 0.16666666666666666,
      "mean_environment_reward": 0.7999999999999998,
      "explorations": 15
    }
  },
  "limitatio

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

results_vcache_hybrid.json
{
  "seed": 0,
  "settings": {
    "workload_path": "src/datasets/example_workload.jsonl",
    "capacity": 32,
    "threshold": 0.85,
    "window": 16
  },
  "method_env_settings": {
    "vCache_official_logic_delta_0.01": {
      "workload_path": "src/datasets/example_workload.jsonl",
      "capacity": 32,
      "threshold": 0.85,
      "window": 16
    },
    "vCache_official_logic_delta_0.02": {
      "workload_path": "src/datasets/example_workload.jsonl",
      "capacity": 32,
      "threshold": 0.85,
      "window": 16
    },
    "vCache_official_logic_delta_0.05": {
      "workload_path": "src/datasets/example_workload.jsonl",
      "capacity": 32,
      "threshold": 0.85,
      "window": 16
    },
    "vCache_official_logic_delta_0.02_lru": {
      "workload_path": "src/datasets/example_workload.jsonl",
      "capacity": 32,
      "threshold": 0.85,
      "window": 16,
      "cache_policy": "lru"
    },
    "FIFO_static": {
      "workload_path": "src/

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>